# 05. Decision-time benchmarks

MicroRTS competitions allocate a fixed per-decision wallclock budget
(typically 100 ms) to every AI. An agent that needs 200 ms per tick is
disqualified even if it would win on accuracy alone, so per-tick
decision time is a first-class metric.

`microrts-agent bench` reports this metric in two modes:

1. **`inference`**: self-play of N agents on the same map, prints
   min/median/max per-tick wall time per agent.
2. **`head2head`**: one RL agent vs one scripted bot, prints decision
   time for both sides.

Numbers are CPU-time, no GPU acceleration assumed.

Prereq: [`00_navigate.ipynb`](00_navigate.ipynb) must be green.

## 1. Self-play inference benchmark

Pit two shipped RL agents against each other (vectorised self-play),
average their per-tick decision time over 3 games. Useful to compare
architectures on equal footing.

In [ ]:
import subprocess

from microrts_agent.paths import PROJECT_ROOT

agents = [
    PROJECT_ROOT / "data" / "agents" / "UECD-SingleMap-Best",  # UNet-Entity-CBAM-Deep, 4.7M params
    PROJECT_ROOT
    / "data"
    / "agents"
    / "GridNet-SingleMap",  # GridNet, 0.84M params (smaller, expected faster)
]

result = subprocess.run(
    [
        "microrts-agent",
        "bench",
        "inference",
        "--agents",
        *(str(a) for a in agents),
        "--games",
        "3",
        "--max-steps",
        "2000",
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=300,
)
print(result.stdout)
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr)

## 2. Head-to-head benchmark

One RL agent vs one scripted bot. Reports decision time for the RL
agent (the only one we control) over 3 games. The scripted bot's time
is the engine's overhead, not something we tune.

In [3]:
result = subprocess.run(
    [
        "microrts-agent",
        "bench",
        "head2head",
        "--agent",
        str(PROJECT_ROOT / "data" / "agents" / "UECD-SingleMap-Best"),
        "--opponent",
        "CoacAI",
        "--games",
        "3",
        "--max-steps",
        "2000",
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=300,
)
print(result.stdout)


  H2H Decision Time Benchmark
  Agent:      /Users/mathisdelsart/Desktop/microrts-drl-uecd/data/agents/UECD-SingleMap-Best
  Opponent:   CoacAI
  Games:      3
  Map:        maps/open_competition/basesWorkers16x16A.xml
  Device:     cpu
  Bot budget: 100 ms

  RL agent: UECD-SingleMap-Best  |  Arch: unet_entity_cbam_deep  |  Params: 4,758,233  |  Device: cpu
  Opponent: CoacAI (Java bot, budget 100 ms)
    Game 1/3: RL mean=29.574 ms, Bot mean=0.218 ms
    Game 2/3: RL mean=28.179 ms, Bot mean=0.179 ms
    Game 3/3: RL mean=27.977 ms, Bot mean=0.162 ms

  SUMMARY (H2H, same game sequence)
  Agent                      Ticks  Mean ms   Med ms   P95 ms
  ----------------------------------------------------------------------
  UECD-SingleMap-Best         2893   27.977   23.874   49.149
  CoacAI                      2890    0.162    0.125    0.354


  CSV saved: outputs/inference_bench/h2h_UECD-SingleMap-Best_vs_CoacAI/h2h_inference_time.csv
  Plot saved: outputs/inference_bench/h2h_UECD-S

## 3. What to read in the output

Both subcommands print a small table per agent with columns like:

- **N**: number of decisions taken (= total ticks across all games).
- **mean / median / p95 / max**: per-tick wallclock in milliseconds.
- **total**: cumulative inference time in seconds.

Key heuristic: the **p95** column is the one to compare to the 100 ms
competition budget. If p95 > 80 ms on the target deployment hardware,
the agent is at risk of timeouts on the slower competition machines.

## Next steps

- The dissertation's full inference benchmark is in
  [`microrts_agent/bench/inference.py`](../microrts_agent/bench/inference.py) and
  [`microrts_agent/bench/head_to_head.py`](../microrts_agent/bench/head_to_head.py).
- For a GPU comparison, add `--device cuda` when the benchmark supports
  it (or run from a GPU SLURM node).
- The competition time budget is configurable per tournament config
  (`timeBudget` field, in ms); see [`03_tournament.ipynb`](03_tournament.ipynb).